# election-spatial-analysis — 01: Data Cleaning

First notebook in the sequence. Numbering starts at `01` rather than following the original
plan's `01_data_collection` / `02_data_cleaning` split, since both source files are already in
hand — there's nothing to "collect."

**What this notebook does:** loads the two OpenHalalan extracts, audits real per-year /
per-position data completeness (rather than assuming a year range is clean), separates
party-list rows from candidate races (they need different handling later), cross-validates the
vote-count file's implied winners against the separate Winners file as a data-quality check, and
writes cleaned Parquet files for the next notebook to build features on.

**What it deliberately does NOT do yet:** PSGC/population joins, feature engineering
(vote share, margin, competitiveness), clustering, or anything geographic — those are separate
notebooks once this one's output is trusted.

**A modeling distinction that matters and is easy to get wrong:** `SENATOR`, `PRESIDENT`, and
`VICE PRESIDENT` are national races reported *per locality* in this file (how did Bangued vote
for president), but the actual winner is decided by the *national sum* across every locality,
not by whichever candidate a given city happened to rank first locally. Local offices
(Governor, Mayor, and so on) are the reverse — each locality's own tally is the complete race.
The cross-validation step below treats these two cases differently on purpose.

Shared logic (text cleaning, locality grouping, the race-feature builder used in `02`) lives in
`src/common.py` rather than being redefined in every notebook.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("..") / "src"))
from common import clean_text, clean_token, first_word, locality_group_cols

pd.set_option("display.max_columns", 30)
print("pandas:", pd.__version__)

pandas: 3.0.2


In [2]:
CONFIG = {
    "vote_counts_path": "../data/raw/NLE_Vote_Counts_2007-2025.csv.gz",
    "winners_path": "../data/raw/NLE_Winners_2004-2025.csv",
    "processed_dir": "../data/processed",
    # a (year, position) combination is kept only if its locality coverage is at least this
    # fraction of the best-covered year for that same position -- see the audit below for why
    # this is data-driven rather than a hardcoded year list.
    "min_coverage_fraction": 0.5,
}

Path(CONFIG["processed_dir"]).mkdir(parents=True, exist_ok=True)
print("Config set.")

Config set.


## Load

In [3]:
votes = pd.read_csv(CONFIG["vote_counts_path"], compression="infer", low_memory=False)
winners = pd.read_csv(CONFIG["winners_path"], low_memory=False)

print("Vote counts:", votes.shape)
print("Winners:", winners.shape)

Vote counts: (2088099, 20)
Winners: (157333, 14)


## Standardize text fields

Trim, uppercase, and collapse repeated whitespace on every locality/name field in both frames,
so joins later aren't silently broken by " Manila" vs "MANILA " vs "Manila". Names with an
apostrophe or an accented letter (ñ) in the vote-counts file also turn up double
HTML-entity-escaped and then uppercased -- e.g. `PEOPLE&AMP;APOS;S` for `PEOPLE'S`,
`BA&AMP;NTILDE;ARES` for `BAÑARES` -- 14,826 rows (0.7%), confirmed by direct inspection.
`clean_text` (in `src/common.py`) undoes that with two passes of `html.unescape`, lowercasing
the entity name in between since the wholesale uppercasing broke case-sensitive entity names
like `&NTILDE;` (though not `&AMP;`, one of a handful recognized either case). This does not fix
actual renamed/reclassified localities (e.g. a municipality that became a city between
elections) — that reconciliation is real work saved for the feature-engineering notebook, once
we know which localities actually need it.

In [4]:
VOTE_TEXT_COLS = ["region", "province", "city", "district", "position", "candidate_name",
                   "last_name", "first_name", "middle_name", "party", "reported_party"]
for col in VOTE_TEXT_COLS:
    votes[col] = clean_text(votes[col])

WINNER_TEXT_COLS = ["Last Name", "First Name", "Middle Name", "Full Name", "Position",
                     "Party", "Province", "City", "Region"]
for col in WINNER_TEXT_COLS:
    winners[col] = clean_text(winners[col])

print("Text fields standardized.")

Text fields standardized.


## Per-year / per-position completeness audit

This is the step that catches problems like "2007 only has 38 of ~1,634 municipalities" or
"2013's Senator race has 33 rows total instead of the ~100,000 every other year has" — both real
issues found by inspection before this notebook was written, and both are the kind of thing that
silently wrecks a clustering feature table if they aren't caught first. Coverage is measured as
the number of distinct localities (`province`+`city`, or `province`+`city`+`district` for House
races) with at least one candidate row, normalized against the best year for that same
position — because a position's own natural denominator (how many localities *should* report it)
differs from every other position's.

In [5]:
def coverage_table(df):
    rows = []
    for position in sorted(df["position"].dropna().unique()):
        sub = df[(df["position"] == position) & (df["is_geographic"])]
        group_cols = locality_group_cols(position)
        for year, year_df in sub.groupby("year"):
            n_localities = year_df[group_cols].drop_duplicates().shape[0]
            rows.append({"position": position, "year": year, "n_localities": n_localities})
    return pd.DataFrame(rows)


coverage = coverage_table(votes)
coverage["max_for_position"] = coverage.groupby("position")["n_localities"].transform("max")
coverage["coverage_fraction"] = coverage["n_localities"] / coverage["max_for_position"]

pivot = coverage.pivot(index="year", columns="position", values="coverage_fraction").round(2)
display(pivot)

flagged = coverage[coverage["coverage_fraction"] < CONFIG["min_coverage_fraction"]]
print(f"\n{len(flagged)} (year, position) combinations fall below "
      f"{CONFIG['min_coverage_fraction']:.0%} of that position's best-covered year:")
display(flagged.sort_values(["position", "year"]))

position,ARMM ASSEMBLYMAN,ARMM REGIONAL GOVERNOR,ARMM REGIONAL VICE GOVERNOR,BARMM MEMBER OF PARLIAMENT,BARMM PARTY REPRESENTATIVE,COUNCILOR,GOVERNOR,MAYOR,"MEMBER, HOUSE OF REPRESENTATIVES",PARTY LIST,PRESIDENT,PROVINCIAL BOARD MEMBER,SENATOR,VICE GOVERNOR,VICE MAYOR,VICE PRESIDENT
year,,,,,,,,,,,,,,,,
2007,NaN,NaN,NaN,NaN,NaN,NaN,0.05,0.02,NaN,NaN,NaN,NaN,NaN,0.00,0.02,NaN
2010,NaN,NaN,NaN,NaN,NaN,0.58,0.70,0.66,0.65,0.65,0.93,0.65,0.93,0.65,0.65,0.93
2013,NaN,NaN,NaN,NaN,NaN,0.91,0.05,0.92,0.11,NaN,NaN,0.05,0.00,0.05,0.92,NaN
2016,1.0,1.0,1.0,NaN,NaN,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
2019,NaN,NaN,NaN,NaN,NaN,1.00,1.00,1.00,1.00,1.00,NaN,1.00,1.00,1.00,1.00,NaN
2022,NaN,NaN,NaN,NaN,NaN,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
2025,NaN,NaN,NaN,1.0,1.0,1.00,1.00,1.00,1.00,1.00,NaN,1.00,1.00,1.00,1.00,NaN



9 (year, position) combinations fall below 50% of that position's best-covered year:


,position,year,n_localities,max_for_position,coverage_fraction
11,GOVERNOR,2007,78,1597,0.048842
13,GOVERNOR,2013,80,1597,0.050094
18,MAYOR,2007,38,1637,0.023213
26,"MEMBER, HOUSE OF REPRESENTATIVES",2013,190,1656,0.114734
40,PROVINCIAL BOARD MEMBER,2013,80,1597,0.050094
46,SENATOR,2013,1,1636,0.000611
51,VICE GOVERNOR,2007,1,1597,0.000626
53,VICE GOVERNOR,2013,75,1597,0.046963
58,VICE MAYOR,2007,36,1637,0.021991


## Apply the coverage threshold

Rows for a flagged (year, position) combination are dropped from the *candidate-race* working
set used downstream, rather than dropping the whole year — a year can be fine for Mayor and
broken for Senator at the same time, as the audit above shows.

In [6]:
flagged_pairs = pd.MultiIndex.from_tuples(
    list(zip(flagged["year"], flagged["position"])), names=["year", "position"]
)
votes_pairs = pd.MultiIndex.from_frame(votes[["year", "position"]])
votes["_flagged_low_coverage"] = votes_pairs.isin(flagged_pairs)
n_dropped = votes["_flagged_low_coverage"].sum()
print(f"Dropping {n_dropped:,} of {len(votes):,} rows ({n_dropped / len(votes):.1%}) "
      f"belonging to a low-coverage (year, position) combination.")

votes_kept = votes[~votes["_flagged_low_coverage"]].drop(columns=["_flagged_low_coverage"]).copy()
print("Remaining rows:", len(votes_kept))
print("Remaining years:", sorted(votes_kept["year"].unique()))

Dropping 3,139 of 2,088,099 rows (0.2%) belonging to a low-coverage (year, position) combination.


Remaining rows: 2084960
Remaining years: [np.int64(2010), np.int64(2013), np.int64(2016), np.int64(2019), np.int64(2022), np.int64(2025)]


## Separate party-list rows

Party-list is a national multi-winner race reported against 100+ parties per locality — over
half the entire file. It needs aggregate treatment (e.g. concentration/top-party share) rather
than one column per party, so it's split out here rather than carried through as an ordinary
"candidate race."

In [7]:
is_partylist = votes_kept["position"] == "PARTY LIST"
votes_partylist = votes_kept[is_partylist].copy()
votes_races = votes_kept[~is_partylist].copy()

print("Party-list rows:", len(votes_partylist))
print("Candidate-race rows:", len(votes_races))
print("\nRemaining positions in the race table:")
print(votes_races["position"].value_counts())

Party-list rows: 1151975
Candidate-race rows: 932985

Remaining positions in the race table:
position
SENATOR                             490138
COUNCILOR                           184288
PROVINCIAL BOARD MEMBER              66764
PRESIDENT                            41618
VICE PRESIDENT                       36916
GOVERNOR                             26036
MAYOR                                22705
VICE MAYOR                           21000
MEMBER, HOUSE OF REPRESENTATIVES     20914
VICE GOVERNOR                        19390
ARMM ASSEMBLYMAN                      1117
BARMM PARTY REPRESENTATIVE             714
ARMM REGIONAL GOVERNOR                 472
ARMM REGIONAL VICE GOVERNOR            472
BARMM MEMBER OF PARLIAMENT             441
Name: count, dtype: int64[pyarrow]


## Winner cross-validation

For local offices, each `(year, province, city[, district], position)` group already *is* the
complete race, so ranking within the group is correct. For national offices
(`PRESIDENT`, `VICE PRESIDENT`, `SENATOR`), votes are summed across every locality first and the
national ranking is what actually decides the winner — ranking within a single city would just
recover that city's local preference, which is a genuinely different (and locality-level
interesting) thing, but not "who won."

Multi-seat positions (`COUNCILOR`, `PROVINCIAL BOARD MEMBER` locally; `SENATOR` nationally) elect
more than one winner per race, so the number of winners to compare against is read from how many
the Winners file actually lists for that group, not assumed to be 1.

In [8]:
NATIONAL_POSITIONS = {"PRESIDENT", "VICE PRESIDENT", "SENATOR"}

# Direct inspection of a matching race (Bangued, Abra, 2010 Councilor) showed the two files
# agree on last name and first given name but disagree on how much of the middle name they
# keep: the vote-counts file truncates it to an initial ("SALVACION B."), the Winners file
# spells it out in full ("SALVACION BEJARIN") -- same person, same election. Keying on the
# full name string breaks on that difference alone. Last name + first given name is stable
# across both files, so the key drops the middle name entirely rather than trying to
# reconcile initial-vs-full-spelling. clean_token also strips diacritics (see common.py),
# since the two files don't agree on whether accents are kept either.
votes_name_parts = (votes_races["candidate_name"].astype("string")
                     .str.split(",", n=1, expand=True).reindex(columns=[0, 1]))
votes_races["_name_key"] = clean_token(votes_name_parts[0]) + "|" + first_word(votes_name_parts[1])
winners["_name_key"] = clean_token(winners["Last Name"]) + "|" + first_word(winners["First Name"])

# Pre-index the (much smaller) winners file ONCE into plain dict lookups, rather than
# re-filtering the full winners frame inside a per-group loop -- with tens of thousands of
# race groups, repeated full-frame boolean filtering is the difference between seconds and
# not finishing. Local lookups are keyed by (Position, Year, Province, City): City alone
# collides whenever two provinces share a municipality name (e.g. Malinao exists in both
# Aklan and Albay, confirmed by inspection), which was silently merging two unrelated races'
# winners into one set. The Winners file has no district column, so multi-district House
# races still only key on city -- a known, disclosed limitation, not fixed here.
local_winner_names = (winners.groupby(["Position", "Year", "Province", "City"])["_name_key"]
                       .apply(set).to_dict())
house_winner_names = (winners[winners["Position"] == "MEMBER, HOUSE OF REPRESENTATIVES"]
                       .groupby(["Position", "Year", "City"])["_name_key"].apply(set).to_dict())
national_winner_names = (winners[winners["Position"].isin(NATIONAL_POSITIONS)]
                          .groupby(["Position", "Year"])["_name_key"].apply(set).to_dict())
# Key -> a readable "Full Name" for display only (the match logic above never uses this).
winner_display_names = winners.groupby("_name_key")["Full Name"].first().to_dict()

match_results = []

# --- Local races: rank within each locality group ---
local_races = votes_races[~votes_races["position"].isin(NATIONAL_POSITIONS)]
for position, pos_df in local_races.groupby("position"):
    group_cols = locality_group_cols(position)
    is_house = position == "MEMBER, HOUSE OF REPRESENTATIVES"
    for keys, group in pos_df.groupby(["year"] + group_cols):
        locality = dict(zip(["year"] + group_cols, keys))
        year, province, city = locality["year"], locality["province"], locality["city"]
        if is_house:
            # No district column in the Winners file to key on -- fall back to city, which
            # under-counts seats (and inflates n_seats) for cities split into >1 district.
            actual = house_winner_names.get((position, year, city))
        else:
            actual = local_winner_names.get((position, year, province, city))
        if not actual:
            continue  # no winners-file record for this exact locality/year -- skip, don't guess
        n_seats = len(actual)
        top_n = group.sort_values("votes", ascending=False).head(n_seats)
        predicted = set(top_n["_name_key"])
        match_results.append({
            "position": position, "matched": len(predicted & actual), "expected": len(actual)
        })

# --- National races: rank on the nationally summed total ---
national_races = votes_races[votes_races["position"].isin(NATIONAL_POSITIONS)]
for position, pos_df in national_races.groupby("position"):
    for year, year_df in pos_df.groupby("year"):
        actual = national_winner_names.get((position, year))
        if not actual:
            continue
        national_totals = year_df.groupby("_name_key")["votes"].sum().sort_values(ascending=False)
        predicted = set(national_totals.head(len(actual)).index)
        match_results.append({
            "position": position, "matched": len(predicted & actual), "expected": len(actual)
        })

match_df = pd.DataFrame(match_results)
summary = match_df.groupby("position").agg(
    groups_checked=("expected", "size"),
    total_expected=("expected", "sum"),
    total_matched=("matched", "sum"),
)
summary["match_rate"] = summary["total_matched"] / summary["total_expected"]
display(summary.sort_values("match_rate"))

overall_rate = match_df["matched"].sum() / match_df["expected"].sum()
print(f"\nOverall winner match rate: {overall_rate:.1%} "
      f"({match_df['matched'].sum():,} / {match_df['expected'].sum():,})")
print("Less than 100% is expected -- covered below, not a bug to chase to zero.")

,groups_checked,total_expected,total_matched,match_rate
position,,,,
COUNCILOR,8717,71909,66104,0.919273
VICE MAYOR,8649,8834,8240,0.932760
MAYOR,8606,8772,8199,0.934679
PRESIDENT,2,2,2,1.000000
SENATOR,4,48,48,1.000000
VICE PRESIDENT,2,2,2,1.000000



Overall winner match rate: 92.2% (82,595 / 89,567)
Less than 100% is expected -- covered below, not a bug to chase to zero.


**Reading the match rate:** under 100% here does not mean the vote-count data is wrong.
The single largest confirmed cause is the Winners file itself: 13,537 of its local-office winner
rows (8.6%), spread across every year from 2001 to 2025, have a missing `City` value even though
`Province` and the candidate are filled in (e.g. Sheryl Capus, Councilor-elect for Malinao, Albay
in 2004/2007/2010, is recorded with `City = NaN` in exactly those years and `City = MALINAO` in
2016/2019 -- same person, same office, inconsistent recording). A row like that can never be
looked up by `(Position, Year, Province, City)` no matter how the name is normalized, which
lines up with the ~8 percentage point gap left after the name- and locality-matching fixes
below. Remaining smaller sources include name-format differences the normalization doesn't
catch (nicknames, suffix handling), the Winners file's outright gaps (no President/Vice-President
winner recorded for 2004 or 2010 at all, confirmed by inspection, so those groups are silently
skipped above rather than counted as mismatches), and genuine ties or post-election
disqualifications that change who is recorded as the "winner" without changing the vote tally.
A handful of concrete mismatches are worth eyeballing before trusting the rate either way:

In [9]:
mismatch_examples = []
for position, pos_df in local_races.groupby("position"):
    group_cols = locality_group_cols(position)
    is_house = position == "MEMBER, HOUSE OF REPRESENTATIVES"
    for keys, group in pos_df.groupby(["year"] + group_cols):
        locality = dict(zip(["year"] + group_cols, keys))
        year, province, city = locality["year"], locality["province"], locality["city"]
        if is_house:
            actual = house_winner_names.get((position, year, city))
        else:
            actual = local_winner_names.get((position, year, province, city))
        if not actual:
            continue
        top1 = group.sort_values("votes", ascending=False).iloc[0]
        if top1["_name_key"] not in actual:
            mismatch_examples.append({
                "year": year, "position": position, "locality": f"{city} ({province})",
                "vote_counts_top1": top1["candidate_name"],
                "winners_file_says": sorted(winner_display_names.get(k, k) for k in actual),
            })
        if len(mismatch_examples) >= 10:
            break
    if len(mismatch_examples) >= 10:
        break

display(pd.DataFrame(mismatch_examples))

,year,position,locality,vote_counts_top1,winners_file_says
0,2010,COUNCILOR,MALINAO (ALBAY),"CAPUS - BILO, SHERYL P.","[BASCO, ALFREDO BASE, CAMU, ALEXIS CASIA, CAS,..."
1,2010,COUNCILOR,BUGASONG (ANTIQUE),"UY KIMPANG, AIDA D.","[ANTOY, GERARDO RIOS, CAPENDIT, BALBINA ADRICU..."
2,2010,COUNCILOR,SAN NICOLAS (BATANGAS),"DE SAGUN, LESTER D.","[ARENAS, JEFFREY SAGUN, BANAAG, BARTOLOME MANA..."
3,2010,COUNCILOR,DONA REMEDIOS TRINIDAD (BULACAN),"DE LEON, MELENCIO S.","[CARPIO, LORNA VALENCIA, CRUZ, HILARIO JOAQUIN..."
4,2010,COUNCILOR,PARACALE (CAMARINES NORTE),"SAN LUIS, EFREN G.","[ASUTILLA, BERNADETTE EPINO, CADIZ, ALELI MANA..."
5,2010,COUNCILOR,NABUA (CAMARINES SUR),"VELITARIO-HAO, MARISSA C.","[ABONAL, ESTEBAN JR. RELATIVO, GRECIA, NOEL FL..."
6,2010,COUNCILOR,SIPOCOT (CAMARINES SUR),"DE LEON, KIMBERLY ANN O.","[ABERGOS, MARCELO BAUTISTA, ALEMANIA, CORAZON ..."
7,2010,COUNCILOR,SIRUMA (CAMARINES SUR),"STA. ROSA, DARWIN S.","[CAMPADO, RODOLFO FABIANO, CRISTOBAL, ANTONIO ..."
8,2010,COUNCILOR,MA AYON (CAPIZ),"DIAZ, JOEL L.","[CONTRERAS, RENE DJ, DESCALZOTA, EDILBERTO BUE..."
9,2010,COUNCILOR,ALFONSO (CAVITE),"DE CASTRO, JUSTINIANO C.","[CAILING, BARTOLOME ROSEL, COSINO, ROMEO RESUR..."


## Locality registry

A deduplicated `province, city, region` table, with `region` back-filled from whichever rows for
that province do have it, since some province-level races (Governor, Vice Governor) leave it
blank in this file's earlier years.

In [10]:
region_lookup = (votes_kept.dropna(subset=["region"])
                  .drop_duplicates(subset=["province"])[["province", "region"]]
                  .set_index("province")["region"])

locality_registry = (votes_races[votes_races["is_geographic"]]
                      [["province", "city"]].drop_duplicates().reset_index(drop=True))
locality_registry["region"] = locality_registry["province"].map(region_lookup)

n_missing_region = locality_registry["region"].isna().sum()
print(f"Locality registry: {len(locality_registry)} unique province/city pairs, "
      f"{n_missing_region} still missing a region after back-fill.")
display(locality_registry.head())

Locality registry: 1865 unique province/city pairs, 0 still missing a region after back-fill.


,province,city,region
0,BASILAN,AKBAR,BARMM
1,BASILAN,AL BARKA,BARMM
2,BASILAN,HADJI MOHAMMAD AJUL,BARMM
3,BASILAN,HADJI MUHTAMAD,BARMM
4,BASILAN,ISABELA,BARMM


## Save cleaned outputs

In [11]:
out_dir = Path(CONFIG["processed_dir"])
votes_races.to_parquet(out_dir / "votes_races_clean.parquet", index=False)
votes_partylist.to_parquet(out_dir / "votes_partylist_clean.parquet", index=False)
locality_registry.to_parquet(out_dir / "locality_registry.parquet", index=False)

print("Saved to", out_dir.resolve())
print(" - votes_races_clean.parquet     ", votes_races.shape)
print(" - votes_partylist_clean.parquet ", votes_partylist.shape)
print(" - locality_registry.parquet     ", locality_registry.shape)

Saved to /tmp/claude-0/-home-claude/8d12b1ae-8bb8-5630-9195-30e2920d12b4/scratchpad/election-spatial-analysis/data/processed
 - votes_races_clean.parquet      (932985, 21)
 - votes_partylist_clean.parquet  (1151975, 20)
 - locality_registry.parquet      (1865, 3)


## Summary

What got cleaned: text fields standardized across both files; low-coverage (year, position)
combinations dropped based on measured completeness rather than an assumed year range;
party-list separated from candidate races; and the vote-count file's implied winners
cross-validated against the independent Winners file, with mismatches inspected rather than
assumed to be zero.

What's still open for the next notebook: locality name reconciliation across LGU boundary
changes and reclassifications (province+city string matching alone doesn't catch a municipality
that became a city between elections), the PSGC/population/urban-rural join, and turning this
cleaned race table into the actual vote-share / margin / competitiveness feature table.